# 02 – Dataset Preparation (Preprocessing)

## Setup & loading raw data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from pathlib import Path
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [ ]:
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)

In [ ]:
bbc_path = RAW_DIR / "bbc-news-summary.csv"
cnn_path = RAW_DIR / "cnn_dailymail_test_dataset.csv"

bbc = pd.read_csv(bbc_path)
cnn = pd.read_csv(cnn_path)

print("BBC Shape:", bbc.shape)
print("CNN Shape:", cnn.shape)

## Text cleaning

In [ ]:
def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
bbc_clean = bbc[["Articles", "Summaries"]].copy()

bbc_clean.columns = ["article", "summary"]

bbc_clean["article"] = bbc_clean["article"].apply(clean_text)
bbc_clean["summary"] = bbc_clean["summary"].apply(clean_text)

# Remove empty records
bbc_clean = bbc_clean[
    (bbc_clean["article"] != "") &
    (bbc_clean["summary"] != "")
]

# Remove duplicate article-summary pairs
bbc_clean = bbc_clean.drop_duplicates(
    subset=["article", "summary"]
)

# Remove repeated articles
bbc_clean = bbc_clean.drop_duplicates(
    subset=["article"]
)

bbc_clean = bbc_clean.reset_index(drop=True)

print("Original BBC:", len(bbc))
print("Clean BBC:", len(bbc_clean))

In [ ]:
cnn_clean = cnn[["article", "highlights"]].copy()

cnn_clean.columns = ["article", "summary"]

cnn_clean["article"] = cnn_clean["article"].apply(clean_text)
cnn_clean["summary"] = cnn_clean["summary"].apply(clean_text)

cnn_clean = cnn_clean[
    (cnn_clean["article"] != "") &
    (cnn_clean["summary"] != "")
]

cnn_clean = cnn_clean.drop_duplicates(
    subset=["article", "summary"]
)

cnn_clean = cnn_clean.drop_duplicates(
    subset=["article"]
)

cnn_clean = cnn_clean.reset_index(drop=True)

print("Original CNN:", len(cnn))
print("Clean CNN:", len(cnn_clean))

## Train / validation / test split and saving

In [27]:
bbc_model_data = bbc_clean[["article", "summary"]].copy()

In [28]:
train_data, temp_data = train_test_split(
    bbc_model_data,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

In [29]:
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Testing:", len(test_data))

print("\nTotal:", len(train_data) + len(validation_data) + len(test_data))

Training: 1700
Validation: 213
Testing: 213

Total: 2126


In [30]:
train_articles = set(train_data["article"])
validation_articles = set(validation_data["article"])
test_articles = set(test_data["article"])

print("Train-Validation Overlap:",
      len(train_articles & validation_articles))

print("Train-Test Overlap:",
      len(train_articles & test_articles))

print("Validation-Test Overlap:",
      len(validation_articles & test_articles))

Train-Validation Overlap: 0
Train-Test Overlap: 0
Validation-Test Overlap: 0


In [31]:
train_data.to_csv(
    SPLIT_DIR / "train.csv",
    index=False
)

validation_data.to_csv(
    SPLIT_DIR / "validation.csv",
    index=False
)

test_data.to_csv(
    SPLIT_DIR / "test.csv",
    index=False
)

cnn_clean[["article", "summary"]].to_csv(
    SPLIT_DIR / "cnn_external_test.csv",
    index=False
)

bbc_clean[["article", "summary"]].to_csv(
    PROCESSED_DIR / "bbc_clean.csv",
    index=False
)

cnn_clean[["article", "summary"]].to_csv(
    PROCESSED_DIR / "cnn_clean.csv",
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!


## Load cleaned data

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

bbc = pd.read_csv(PROCESSED_DIR / "bbc_clean.csv")
cnn = pd.read_csv(PROCESSED_DIR / "cnn_clean.csv")

print("BBC:", bbc.shape)
print("CNN:", cnn.shape)

display(bbc.head(3))
display(cnn.head(3))

BBC: (2126, 2)
CNN: (11488, 2)


,article,summary
0,Budget to set scene for election..Gordon Brown...,- Increase in the stamp duty threshold from £6...
1,Army chiefs in regiments decision..Military ch...,"""They are very much not for the good and will ..."
2,Howard denies split over ID cards..Michael How...,Michael Howard has denied his shadow cabinet w...


,article,summary
0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...
1,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b..."
2,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...


## Sanity checks

In [2]:
for name, dataset in [("BBC", bbc), ("CNN", cnn)]:

    print(f"\n{name} Dataset")
    print("Columns:", dataset.columns.tolist())
    print("Missing values:\n", dataset.isnull().sum())
    print("Duplicate articles:", dataset["article"].duplicated().sum())


BBC Dataset
Columns: ['article', 'summary']
Missing values:
 article    0
summary    0
dtype: int64
Duplicate articles: 0

CNN Dataset
Columns: ['article', 'summary']
Missing values:
 article    0
summary    0
dtype: int64
Duplicate articles: 0


## Tokenizer & token-length analysis

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)

c:\Users\Kowshik\text-summarization-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Kowshik\text-summarization-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kowshik\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to

Tokenizer loaded: t5-small


In [4]:
def get_token_length(text):

    tokens = tokenizer(
        str(text),
        truncation=False
    )

    return len(tokens["input_ids"])

In [5]:
bbc["article_tokens"] = bbc["article"].apply(get_token_length)
bbc["summary_tokens"] = bbc["summary"].apply(get_token_length)

cnn["article_tokens"] = cnn["article"].apply(get_token_length)
cnn["summary_tokens"] = cnn["summary"].apply(get_token_length)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (708 > 512). Running this sequence through the model will result in indexing errors


In [6]:
print("BBC Article Token Statistics:")
display(bbc["article_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

print("BBC Summary Token Statistics:")
display(bbc["summary_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

BBC Article Token Statistics:


count    2126.000000
mean      533.279398
std       327.498191
min       117.000000
50%       459.000000
90%       846.500000
95%      1013.750000
99%      1391.000000
max      6002.000000
Name: article_tokens, dtype: float64

BBC Summary Token Statistics:


count    2126.000000
mean      238.852305
std       155.181900
min        52.000000
50%       205.000000
90%       388.000000
95%       450.000000
99%       627.250000
max      2925.000000
Name: summary_tokens, dtype: float64

In [7]:
print("CNN Article Token Statistics:")
display(cnn["article_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

print("CNN Summary Token Statistics:")
display(cnn["summary_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

CNN Article Token Statistics:


count    11488.000000
mean       969.201950
std        497.842275
min         73.000000
50%        870.000000
90%       1689.000000
95%       1973.650000
99%       2436.000000
max       3688.000000
Name: article_tokens, dtype: float64

CNN Summary Token Statistics:


count    11488.000000
mean        79.300313
std         32.373987
min         12.000000
50%         74.000000
90%        116.000000
95%        135.000000
99%        174.000000
max        968.000000
Name: summary_tokens, dtype: float64

In [1]:
from transformers import AutoTokenizer

MODEL_NAME = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

c:\Users\Kowshik\my-project\text-summarization-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded successfully!


In [2]:
sample_text = "Artificial intelligence is changing the world."

tokens = tokenizer(sample_text)

print("Original text:", sample_text)
print("Token IDs:", tokens["input_ids"])
print("Token count:", len(tokens["input_ids"]))

Original text: Artificial intelligence is changing the world.
Token IDs: [24714, 6123, 19, 2839, 8, 296, 5, 1]
Token count: 8


In [3]:
def get_article_token_length(text):
    text = "summarize: " + str(text)
    return len(tokenizer(text, truncation=False)["input_ids"])


def get_summary_token_length(text):
    return len(tokenizer(str(text), truncation=False)["input_ids"])

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\Kowshik\my-project\text-summarization-system")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Processed folder exists:", PROCESSED_DIR.exists())

print("\nAvailable CSV files:")
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name)

Current notebook location:
c:\Users\Kowshik\my-project\text-summarization-system\notebooks

Your user folder:
C:\Users\Kowshik

Possible project locations:

c:\Users\Kowshik\my-project\text-summarization-system\notebooks
Exists: True
   01_data_exploration.ipynb
   02_dataset_preparation.ipynb

C:\Users\Kowshik\my-project
Exists: True
   event-managment
   learning-web-technologies-spring2025-2026-sec-q
   login-registration
   Student-management
   text-summarization-system

C:\Users\Kowshik\my-project\text-summarization-system
Exists: True
   .venv
   data
   notebooks
   outputs
   requirements.txt
   src


In [7]:
bbc["article_tokens"] = bbc["article"].apply(get_article_token_length)
bbc["summary_tokens"] = bbc["summary"].apply(get_summary_token_length)

cnn["article_tokens"] = cnn["article"].apply(get_article_token_length)
cnn["summary_tokens"] = cnn["summary"].apply(get_summary_token_length)

print("Token length calculation completed.")

NameError: name 'bbc' is not defined

In [5]:
percentiles = [0.50, 0.90, 0.95, 0.99]

print("BBC Article Token Lengths")
display(bbc["article_tokens"].describe(percentiles=percentiles))

print("BBC Summary Token Lengths")
display(bbc["summary_tokens"].describe(percentiles=percentiles))

print("CNN Article Token Lengths")
display(cnn["article_tokens"].describe(percentiles=percentiles))

print("CNN Summary Token Lengths")
display(cnn["summary_tokens"].describe(percentiles=percentiles))

BBC Article Token Lengths


NameError: name 'bbc' is not defined